In [ ]:
from dotenv import load_dotenv
import sys
from pathlib import Path

# Get the parent directory of the current working directory
parent_dir = Path.cwd().parent.parent

# Add the parent directory's path to sys.path
# sys.path requires strings, so we convert the Path object
sys.path.append(str(parent_dir))

load_dotenv()

In [ ]:
import json
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from src.dataset.load_data_soda import SODADataLoader
from src.dataset.n2d_translate import N2DTranslate

In [ ]:
soda_dataset_obj = SODADataLoader(
    percent_of_all_splits=0.1,
    min_story_length=20,
    max_story_length=250,
    max_dialogue_length=250,
    show_dataset_info_after_load=False,
    unroll_dialogue_and_speakers=False,
    join_dialogue_and_speakers=False,
)
soda_dataset_obj.show_dataset_info(show_features=True, show_word_counts=True)
soda_ds = soda_dataset_obj.dataset

In [ ]:
all_narratives = {}
for split in soda_ds.keys():
    print(f"Processing split: {split}")
    all_narratives[split] = soda_ds[split]["narrative"]

In [ ]:
all_dialogues = {}
dialogue_counts = {}


def get_len(example):
    return {"len_col": len(example["dialogue"])}


for split in soda_ds.keys():
    print(f"Processing split: {split}")
    dialogues = soda_ds[split]["dialogue"]
    all_dialogues[split] = []
    for dialogue_group in dialogues:
        all_dialogues[split].extend(dialogue_group)
    dialogue_counts[split] = soda_ds[split].map(get_len)["len_col"]

In [ ]:
n2d_translator = N2DTranslate(
    attn_implementation="flash_attention_2",
)

In [ ]:
translated_narratives = {}
for split in all_narratives.keys():
    print(f"Translating narratives for split: {split}")
    translated_narratives[split] = n2d_translator.translate_texts(
        all_narratives[split],
        batch_size=32
    )

In [ ]:
translated_dialogues = {}
for split in all_dialogues.keys():
    print(f"Translating dialogues for split: {split}")
    translated_dialogues[split] = n2d_translator.translate_texts(
        all_dialogues[split],
        batch_size=16
    )

In [ ]:
temp_trans_dialogues = translated_dialogues.copy()
translated_dialogues_groups = {}
for split in dialogue_counts.keys():
    translated_dialogues_groups[split] = []
    for i in tqdm((dialogue_counts[split]), desc=f"Processing dialogues for {split} split"):
        translated_dialogues_groups[split].append(
            temp_trans_dialogues[split][:i])
        temp_trans_dialogues[split] = temp_trans_dialogues[split][i:]

In [ ]:
mapping_path = hf_hub_download(
    repo_id="abirmondalind/soda_bengali_small",
    filename="name_mapping.json",
    repo_type="dataset"
)

In [ ]:
translated_speakers = {}
with open(mapping_path, "r", encoding='utf-8') as f:
    name_mapping = json.load(f)
    for split in soda_ds.keys():
        translated_speakers[split] = []
        for speakers in soda_ds[split]['speakers']:
            translated_speakers[split].append(
                [name_mapping.get(speaker, speaker) for speaker in speakers])

In [ ]:
for i, split in enumerate(soda_ds.keys()):
    with open(f"n2d_eng_to_ben_{split}.jsonl", "w", encoding='utf-8') as f:
        for j, row in tqdm(enumerate(soda_ds[split]), total=len(soda_ds[split]), desc=f"Writing records for {split} split"):
            record = {
                "index": j,
                "original_narrative": soda_ds[split][j]['narrative'],
                "translated_narrative": translated_narratives[split][j],
                "original_dialogue": soda_ds[split][j]['dialogue'],
                "translated_dialogue": translated_dialogues_groups[split][j],
                "original_speakers": soda_ds[split][j]['speakers'],
                "translated_speakers": translated_speakers[split][j]
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")